# MiniTorch: GPU-Accelerated Transformer Framework - Project Demo

This notebook provides a comprehensive demonstration of the MiniTorch framework, showcasing features implemented across four assignments:

1. **Assignment 1**: CUDA Tensor Operations (map, zip, reduce, matrix multiply)
2. **Assignment 2**: Automatic Differentiation
3. **Assignment 3**: Transformer Architecture
4. **Assignment 4**: Optimized CUDA Kernels (fused softmax and layernorm)

Each section demonstrates working code examples with explanations and performance benchmarks.


## Environment Setup

First, we need to set up the environment and install dependencies.


### Step 1: Clone Repository and Install Dependencies

**Note**: If running on Google Colab, uncomment and run the following cells.


In [ ]:
#Clone repository (uncomment if on Colab)
!git clone https://github.com/ppaleja/GPU-accl-transfrmr-framwrk-in-Minitorch.git
%cd llmsys_f25_hw4


fatal: destination path 'llmsys_f25_hw4' already exists and is not an empty directory.
/content/llmsys_f25_hw4


In [2]:
# Install other dependencies (uncomment if on Colab)
!pip install -r requirements.txt
!pip install -r requirements.extra.txt
!pip install -Ue .


Obtaining file:///content/llmsys_f25_hw4
  Preparing metadata (setup.py) ... done
  Attempting uninstall: minitorch
    Found existing installation: minitorch 0.4
    Uninstalling minitorch-0.4:
      Successfully uninstalled minitorch-0.4
  Running setup.py develop for minitorch


In [3]:
# Install PyTorch with CUDA support (uncomment if on Colab)
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124


Looking in indexes: https://download.pytorch.org/whl/cu124


### Step 2: Compile CUDA Kernels

This compiles the custom CUDA kernels for tensor operations, softmax, and layernorm.


In [4]:
# Compile CUDA kernels
!bash compile_cuda.sh


### Step 3: Verify GPU Availability


In [5]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA Version:", torch.version.cuda)
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA not available. Some features will not work.")


PyTorch Version: 2.5.1+cu124
CUDA Available: True
CUDA Version: 12.4
GPU Device: Tesla T4


### Step 4: Import MiniTorch and Required Modules


In [6]:
import minitorch
from minitorch import Tensor, TensorBackend
from minitorch.cuda_kernel_ops import CudaKernelOps
import numpy as np
import time

print("MiniTorch imported successfully!")


MiniTorch imported successfully!


In [7]:
# Create CUDA backend
backend = TensorBackend(CudaKernelOps)
print("CUDA backend created successfully!")


CUDA backend created successfully!


---

## Assignment 1: CUDA Tensor Operations

Assignment 1 implemented custom CUDA kernels for fundamental tensor operations. These kernels provide GPU-accelerated computation for:
- **Map**: Element-wise unary operations
- **Zip**: Element-wise binary operations  
- **Reduce**: Aggregation along dimensions
- **Matrix Multiply**: Optimized matmul with shared memory

All kernels are implemented in `src/combine.cu`.


### 1.1 Map Operations

Map operations apply a unary function element-wise to tensors. Examples include ReLU, sigmoid, log, exp, negation, etc.


In [8]:
# Create a tensor with mixed positive and negative values
x = minitorch.tensor([1.0, -2.0, 3.0, -4.0, 5.0], backend=backend)
print("Input tensor:", x.to_numpy())

# Apply ReLU (max(0, x))
y = x.relu()
print("After ReLU:", y.to_numpy())

# Apply sigmoid
z = x.sigmoid()
print("After sigmoid:", z.to_numpy())


Input tensor: [ 1. -2.  3. -4.  5.]
After ReLU: [1. 0. 3. 0. 5.]
After sigmoid: [0.73105854 0.11920292 0.95257413 0.01798621 0.9933072 ]


**Explanation**: The CUDA map kernel launches one thread per output element. Each thread:
1. Calculates its global position using blockIdx and threadIdx
2. Converts the flat position to multidimensional indices using strides
3. Applies the operation function
4. Writes the result to output memory


In [9]:
# More map examples
a = minitorch.tensor([1.0, 2.0, 3.0, 4.0], backend=backend)

print("Original:", a.to_numpy())
print("Negation:", (-a).to_numpy())
print("Exponential:", a.exp().to_numpy())
print("Logarithm:", a.log().to_numpy())


Original: [1. 2. 3. 4.]
Negation: [-1. -2. -3. -4.]
Exponential: [ 2.7182817  7.389056  20.085537  54.59815  ]
Logarithm: [9.9999954e-07 6.9314766e-01 1.0986127e+00 1.3862946e+00]


### 1.2 Zip Operations

Zip operations apply a binary function element-wise to pairs of tensors.


In [10]:
# Create two tensors
a = minitorch.tensor([1.0, 2.0, 3.0, 4.0], backend=backend)
b = minitorch.tensor([5.0, 6.0, 7.0, 8.0], backend=backend)

print("Tensor a:", a.to_numpy())
print("Tensor b:", b.to_numpy())
print()

# Element-wise operations
print("a + b:", (a + b).to_numpy())
print("a * b:", (a * b).to_numpy())
print("a < b:", (a < b).to_numpy())
print("a == b:", (a == b).to_numpy())


Tensor a: [1. 2. 3. 4.]
Tensor b: [5. 6. 7. 8.]

a + b: [ 6.  8. 10. 12.]
a * b: [ 5. 12. 21. 32.]
a < b: [1. 1. 1. 1.]
a == b: [0. 0. 0. 0.]


**Broadcasting Support**: The zip kernel supports broadcasting for tensors of different shapes.


In [11]:
# Broadcasting example
x = minitorch.tensor([[1.0, 2.0, 3.0]], backend=backend)  # Shape: (1, 3)
y = minitorch.tensor([[1.0], [2.0], [3.0]], backend=backend)  # Shape: (3, 1)

print("x shape:", x.shape)
print("y shape:", y.shape)
print()

z = x + y  # Broadcasting to (3, 3)
print("x + y (broadcasted):")
print(z.to_numpy())
print("Result shape:", z.shape)


x shape: (1, 3)
y shape: (3, 1)

x + y (broadcasted):
[[2. 3. 4.]
 [3. 4. 5.]
 [4. 5. 6.]]
Result shape: (3, 3)


### 1.3 Reduce Operations

Reduce operations aggregate elements along specified dimensions using a binary function.


In [12]:
# Create a 2D tensor
x = minitorch.tensor([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0]], backend=backend)
print("Original tensor:")
print(x.to_numpy())
print()

# Sum along dimension 0 (across rows)
sum_dim0 = x.sum(0)
print("Sum along dim 0:", sum_dim0.to_numpy())

# Sum along dimension 1 (across columns)
sum_dim1 = x.sum(1)
print("Sum along dim 1:", sum_dim1.to_numpy())

# Mean reduction
mean_all = x.mean()
print("Mean of all elements:", mean_all.to_numpy())


Original tensor:
[[1. 2. 3.]
 [4. 5. 6.]]

Sum along dim 0: [[5. 7. 9.]]
Sum along dim 1: [[ 6.]
 [15.]]
Mean of all elements: [3.5]


**Explanation**: The reduce kernel uses shared memory for efficient block-level reduction:
1. Threads load elements into shared memory
2. Threads cooperatively reduce using a parallel reduction tree
3. Final result is written to global memory


### 1.4 Matrix Multiplication

The matmul kernel implements optimized matrix multiplication using shared memory tiling (TILE=32).


In [13]:
# Create two matrices
A = minitorch.tensor([[1.0, 2.0, 3.0],
                       [4.0, 5.0, 6.0]], backend=backend)  # 2x3
B = minitorch.tensor([[7.0, 8.0],
                       [9.0, 10.0],
                       [11.0, 12.0]], backend=backend)  # 3x2

print("Matrix A (2x3):")
print(A.to_numpy())
print()
print("Matrix B (3x2):")
print(B.to_numpy())
print()

C = A @ B  # Matrix multiplication
print("A @ B (2x2):")
print(C.to_numpy())


Matrix A (2x3):
[[1. 2. 3.]
 [4. 5. 6.]]

Matrix B (3x2):
[[ 7.  8.]
 [ 9. 10.]
 [11. 12.]]

A @ B (2x2):
[[ 58.  64.]
 [139. 154.]]


**Optimization**: The kernel uses shared memory to:
- Load tiles of matrices A and B into shared memory
- Minimize global memory accesses
- Enable efficient reuse of data across threads
- Achieve significant speedup over naive implementation


---

## Assignment 2: Automatic Differentiation

Assignment 2 implemented the automatic differentiation engine, enabling gradient computation for neural network training. Key components:
- **Topological Sort**: Orders computation graph for backpropagation
- **Backpropagation**: Computes gradients via chain rule

Implementation in `minitorch/autodiff.py`.


### 2.1 Basic Autodiff with Scalars


In [14]:
from minitorch import Scalar

# Create scalar variables
x = Scalar(3.0)
y = Scalar(4.0)

# Build computation graph
z = x * y + x
print(f"z = x * y + x = {z.data}")

# Compute gradients
z.backward()
print(f"dz/dx = {x.derivative}")  # Should be y + 1 = 5
print(f"dz/dy = {y.derivative}")  # Should be x = 3


z = x * y + x = 15.0
dz/dx = 5.0
dz/dy = 3.0


### 2.2 Autodiff with Tensors

The same autodiff engine works with tensors for neural network training.


In [15]:
# Create tensors with gradient tracking
x = minitorch.tensor([[1.0, 2.0], [3.0, 4.0]], backend=backend, requires_grad=True)
y = minitorch.tensor([[5.0, 6.0], [7.0, 8.0]], backend=backend, requires_grad=True)

print("x:")
print(x.to_numpy())
print("y:")
print(y.to_numpy())

# Forward pass
z = x * y + x
print("\nz = x * y + x:")
print(z.to_numpy())

# Backward pass
z.sum().backward()

print("\nGradients:")
print("dx (should be y + 1):")
print(x.grad.to_numpy() if x.grad else "None")
print("dy (should be x):")
print(y.grad.to_numpy() if y.grad else "None")


x:
[[1. 2.]
 [3. 4.]]
y:
[[5. 6.]
 [7. 8.]]

z = x * y + x:
[[ 6. 14.]
 [24. 36.]]

Gradients:
dx (should be y + 1):
[[6. 7.]
 [8. 9.]]
dy (should be x):
[[1. 2.]
 [3. 4.]]


### 2.3 Neural Network Modules

Assignment 2 also introduced modular neural network components.


In [18]:
from minitorch import Parameter, Linear

# Create a simple linear layer
linear = Linear(3, 2, bias=True, backend=backend)

# Forward pass
x = minitorch.tensor([[1.0, 2.0, 3.0]], backend=backend)
y = linear(x)

print("Input shape:", x.shape)
print("Output shape:", y.shape)
print("Output:")
print(y.to_numpy())

Input shape: (1, 3)
Output shape: (1, 2)
Output:
[[-0.07887308  0.12703478]]


---

## Assignment 3: Transformer Architecture

Assignment 3 implemented a complete decoder-only transformer architecture (GPT-2 style) with:
- Multi-head attention
- Position-wise feed-forward networks
- Layer normalization
- Positional embeddings
- Full decoder language model

Implementation in `minitorch/modules_transfomer.py` and `minitorch/modules_basic.py`.


### 3.1 Embedding Layer

Maps discrete token IDs to continuous vector representations.


In [17]:
from minitorch import Embedding

# Create embedding layer
vocab_size = 100
embedding_dim = 64
embedding = Embedding(vocab_size, embedding_dim, backend=backend)

# Create token IDs
token_ids = minitorch.tensor([[1, 2, 3, 4]], backend=backend)  # Batch size 1, sequence length 4
print("Token IDs:", token_ids.to_numpy())

# Get embeddings
embeddings = embedding(token_ids)
print("\nEmbeddings shape:", embeddings.shape)  # Should be (1, 4, 64)
print("First token embedding (first 10 dims):", embeddings.to_numpy()[0, 0, :10])


Token IDs: [[1. 2. 3. 4.]]

Embeddings shape: (1, 4, 64)
First token embedding (first 10 dims): [-0.02717306  0.01395027 -0.03160924 -0.02655143 -0.00933728 -0.03332524
 -0.00022865  0.04556605 -0.00827673 -0.00918569]


### 3.2 Multi-Head Attention

The core of the transformer: scaled dot-product attention with multiple heads.


In [25]:
from minitorch import MultiHeadAttention, Tensor

# Create multi-head attention layer
n_embd = 64
n_head = 4
mha = MultiHeadAttention(n_embd, n_head, causal=True, backend=backend)

# Create input
batch_size, seq_len = 2, 8

# Create the numpy array
np_array_data = np.random.randn(batch_size, seq_len, n_embd).astype(np.float32)

# Explicitly create the minitorch tensor using Tensor.make
# This ensures the shape is correctly passed to the TensorData constructor.
x = Tensor.make(
    np_array_data.flatten().tolist(), # data as a flat list
    shape=(batch_size, seq_len, n_embd), # explicit shape tuple
    backend=backend
)
print("Input shape:", x.shape)

# Forward pass
output = mha(x)
print("Output shape:", output.shape)
print("\nMulti-head attention applies Q, K, V projections,")
print("computes scaled dot-product attention, and projects back.")

Input shape: (2, 8, 64)
Output shape: (2, 8, 64)

Multi-head attention applies Q, K, V projections,
computes scaled dot-product attention, and projects back.


**Attention Mechanism**:
1. Project input to Query (Q), Key (K), Value (V)
2. Reshape to separate heads
3. Compute attention scores: `scores = Q @ K^T / sqrt(d_k)`
4. Apply softmax to get attention weights
5. Compute weighted sum of values: `output = softmax(scores) @ V`
6. Concatenate heads and apply output projection


### 3.3 Transformer Layer

Combines multi-head attention with feed-forward network, layer normalization, and residual connections.


In [27]:
from minitorch import TransformerLayer, Tensor

# Create transformer layer
n_embd = 64
n_head = 4
transformer_layer = TransformerLayer(n_embd, n_head, backend=backend)

# Forward pass
np_array_data = np.random.randn(2, 8, n_embd).astype(np.float32)
x = Tensor.make(
    np_array_data.flatten().tolist(),
    shape=(2, 8, n_embd),
    backend=backend
)
output = transformer_layer(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("\nTransformer layer = LayerNorm -> Attention -> Residual ->")
print("                    LayerNorm -> FFN -> Residual")

Input shape: (2, 8, 64)
Output shape: (2, 8, 64)

Transformer layer = LayerNorm -> Attention -> Residual ->
                    LayerNorm -> FFN -> Residual


### 3.4 Full Decoder Language Model

The complete transformer decoder for language modeling.


In [43]:
from minitorch import DecoderLM, Tensor

# Create small decoder model
vocab_size = 100
n_embd = 64
n_head = 4
# num_layers = 2 # This parameter is not accepted by DecoderLM
max_len = 128

model = DecoderLM(
    n_vocab=vocab_size,
    n_embd=n_embd,
    n_head=n_head,
    n_positions=max_len, # Use n_positions for max_len
    # num_layers=num_layers, # Remove this argument
    backend=backend
)

# Forward pass
batch_size, seq_len = 2, 10
input_ids_np = np.random.randint(0, vocab_size, (batch_size, seq_len)).astype(np.float32)
input_ids = Tensor.make(
    input_ids_np.flatten().tolist(),
    shape=(batch_size, seq_len),
    backend=backend
)

print("Input IDs shape:", input_ids.shape)

logits = model(input_ids)
print("Logits shape:", logits.shape)  # Should be (batch_size, seq_len, vocab_size)
print("\nThe model outputs probability distributions over the vocabulary for each position.")

Input IDs shape: (2, 10)
Logits shape: (2, 10, 100)

The model outputs probability distributions over the vocabulary for each position.


---

## Assignment 4: Optimized CUDA Kernels

Assignment 4 implemented fused CUDA kernels for softmax and layernorm operations, achieving significant speedups:
- **Fused Softmax**: Forward ~6.5×, Backward ~0.5× vs PyTorch
- **Fused LayerNorm**: Forward ~15.8×, Backward ~3.7× vs PyTorch

Implementation in `src/softmax_kernel.cu` and `src/layernorm_kernel.cu`.


### 4.1 Fused Softmax Kernel

The fused softmax kernel combines max-finding, exp, sum, and normalization into a single kernel, reducing memory bandwidth.


In [46]:
from minitorch import Attn_Softmax, Tensor

# Create attention scores (QK^T / sqrt(d_k))
batch_size, n_head, seq_len = 2, 4, 32
scores_np = np.random.randn(batch_size, n_head, seq_len, seq_len).astype(np.float32)
scores = Tensor.make(
    scores_np.flatten().tolist(),
    shape=(batch_size, n_head, seq_len, seq_len),
    backend=backend
)

# Create attention mask (for causal masking)
mask_np = np.triu(np.ones((batch_size, n_head, seq_len, seq_len)) * -1e8, 1).astype(np.float32)
mask = Tensor.make(
    mask_np.flatten().tolist(),
    shape=(batch_size, n_head, seq_len, seq_len),
    backend=backend
)

print("Scores shape:", scores.shape)
print("Mask shape:", mask.shape)

# Apply fused softmax
attn_weights = Attn_Softmax.apply(scores, mask)
print("Attention weights shape:", attn_weights.shape)
print("\nFused softmax computes: softmax(scores + mask)")
print("Sum along last dim (should be ~1.0):", attn_weights.sum(3).to_numpy()[0, 0, :5])

Scores shape: (2, 4, 32, 32)
Mask shape: (2, 4, 32, 32)
Attention weights shape: (2, 4, 32, 32)

Fused softmax computes: softmax(scores + mask)
Sum along last dim (should be ~1.0): [[1.]
 [1.]
 [1.]
 [1.]
 [1.]]


**Softmax Optimization Details**:
- **Short sequences (<32)**: Warp-level reduction using warp shuffle primitives
- **Long sequences**: Block-level reduction using CUB library
- **Three-stage algorithm**:
  1. Find max for numerical stability
  2. Compute exp and sum
  3. Normalize by sum
- Fuses mask addition to avoid separate kernel launch


### 4.2 Fused LayerNorm Kernel

The fused layernorm kernel computes mean and variance concurrently using the formula: σ² = E[x²] - E[x]²


In [53]:
from minitorch import LayerNorm1d, Tensor

# Create layer normalization
hidden_dim = 64
ln_eps = 1e-5 # Define epsilon for LayerNorm
ln = LayerNorm1d(hidden_dim, eps=ln_eps, backend=backend)

# Input tensor
batch_size, seq_len = 4, 16
x = Tensor.make(
    np.random.randn(batch_size, seq_len, hidden_dim).astype(np.float32).flatten().tolist(),
    shape=(batch_size, seq_len, hidden_dim),
    backend=backend
)

print("Input shape:", x.shape)

# Apply layer normalization
x_reshaped = x.view(batch_size * seq_len, hidden_dim)
output_reshaped = ln(x_reshaped)
output = output_reshaped.view(batch_size, seq_len, hidden_dim)

print("Output shape:", output.shape)

# Verify normalization (mean ≈ 0, std ≈ 1)
print("\nOutput statistics (per feature):")
print("Mean (should be ~0):", output.mean().to_numpy())
# Calculate standard deviation manually as Tensor does not have a std method
std_dev = np.std(output.to_numpy())
print("Std (should be ~1):", std_dev)

Input shape: (4, 16, 64)
Output shape: (4, 16, 64)

Output statistics (per feature):
Mean (should be ~0): [6.1118044e-09]
Std (should be ~1): 0.9999949


**LayerNorm Optimization Details**:
- **float4 vectorization**: Processes 4 elements at a time for memory efficiency
- **Single-pass algorithm**: Computes E[x] and E[x²] concurrently
- **Backward pass split**: Separate kernels for input gradients and parameter gradients
- **Warp shuffle**: Used for efficient reduction without shared memory in some cases


### 4.3 Performance Benchmarking

Let's benchmark the fused kernels against PyTorch implementations.


In [54]:
# Note: These benchmarks require the kernel test files
# Here we demonstrate the speedup expectations

print("Expected Speedups vs PyTorch:\n")
print("| Kernel       | Operation | Speedup |")
print("|--------------|-----------|---------|")
print("| Softmax      | Forward   | ~6.5×   |")
print("| Softmax      | Backward  | ~0.5×   |")
print("| LayerNorm    | Forward   | ~15.8×  |")
print("| LayerNorm    | Backward  | ~3.7×   |")
print()
print("To run actual benchmarks:")
print("  python kernel_tests/test_softmax_fw.py")
print("  python kernel_tests/test_softmax_bw.py")
print("  python kernel_tests/test_layernorm_fw.py")
print("  python kernel_tests/test_layernorm_bw.py")


Expected Speedups vs PyTorch:

| Kernel       | Operation | Speedup |
|--------------|-----------|---------|
| Softmax      | Forward   | ~6.5×   |
| Softmax      | Backward  | ~0.5×   |
| LayerNorm    | Forward   | ~15.8×  |
| LayerNorm    | Backward  | ~3.7×   |

To run actual benchmarks:
  python kernel_tests/test_softmax_fw.py
  python kernel_tests/test_softmax_bw.py
  python kernel_tests/test_layernorm_fw.py
  python kernel_tests/test_layernorm_bw.py


In [55]:
# Comment this out to run the benchmarks
# !python kernel_tests/test_softmax_fw.py
# !python kernel_tests/test_softmax_bw.py
# !python kernel_tests/test_layernorm_fw.py
# !python kernel_tests/test_layernorm_bw.py

>>>>>>>>>>>>>>>>>>>>>>test_launch_attn_softmax, ntest [0], dtype [torch.float32]:
(batch_size, nhead, from_len, to_len, is_dec_self_attn, is_dec_self_attn_infer): (1, 8, 115, 282, False, False)
Run baseline...
Run custom...
Compare the results of custom and baseline...
Test passed. Time of custom/baseline (ms): 1.201 / 11.295, speedup: 9.406
>>>>>>>>>>>>>>>>>>>>>>test_launch_attn_softmax, ntest [1], dtype [torch.float32]:
(batch_size, nhead, from_len, to_len, is_dec_self_attn, is_dec_self_attn_infer): (2, 8, 229, 105, False, False)
Run baseline...
Run custom...
Compare the results of custom and baseline...
Test passed. Time of custom/baseline (ms): 1.503 / 14.936, speedup: 9.940
>>>>>>>>>>>>>>>>>>>>>>test_launch_attn_softmax, ntest [2], dtype [torch.float32]:
(batch_size, nhead, from_len, to_len, is_dec_self_attn, is_dec_self_attn_infer): (10, 8, 90, 55, False, False)
Run baseline...
Run custom...
Compare the results of custom and baseline...
Test passed. Time of custom/baseline (ms): 

### 4.4 Integration into Transformer

The fused kernels are integrated into the transformer architecture for end-to-end speedup.


In [56]:
from minitorch import MultiHeadAttention, TransformerLayer

# Create transformer components with fused kernels enabled
n_embd = 64
n_head = 4

# Multi-head attention with fused softmax
mha_fused = MultiHeadAttention(n_embd, n_head, backend=backend, use_fused_kernel=True)

# Transformer layer with fused layernorm
layer_fused = TransformerLayer(n_embd, n_head, backend=backend, use_fused_kernel=True)

print("Transformer components created with fused kernels enabled")
print("\nWhen training with fused kernels:")
print("  python project/run_machine_translation.py --use-fused-kernel True")
print("\nExpected end-to-end speedup: ~1.1×")
print("(Modest due to Amdahl's law - softmax/layernorm are small fraction of total time)")


Transformer components created with fused kernels enabled

When training with fused kernels:
  python project/run_machine_translation.py --use-fused-kernel True

Expected end-to-end speedup: ~1.1×
(Modest due to Amdahl's law - softmax/layernorm are small fraction of total time)


---

## Running Tests

The repository includes comprehensive tests for all components.


### Unit Tests


In [ ]:
# Run autodiff tests
!pytest tests/test_autodiff.py -v

# Run tensor operation tests
!pytest tests/test_tensor_general.py -v

# Run transformer module tests
!pytest tests/test_modules_transformer.py -v


============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-7.1.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
hypothesis profile 'default' -> database=DirectoryBasedExampleDatabase('/content/llmsys_f25_hw4/.hypothesis/examples')
rootdir: /content/llmsys_f25_hw4
plugins: hypothesis-6.54.0, env-0.6.2, langsmith-0.4.40, anyio-4.11.0, typeguard-4.4.4
collected 8 items                                                              

tests/test_autodiff.py::test_chain_rule1 PASSED                          [ 12%]
tests/test_autodiff.py::test_chain_rule2 PASSED                          [ 25%]
tests/test_autodiff.py::test_chain_rule3 PASSED                          [ 37%]
tests/test_autodiff.py::test_chain_rule4 PASSED                          [ 50%]
tests/test_autodiff.py::test_backprop1 PASSED                            [ 62%]
tests/test_autodiff.py::test_backprop2 PASSED                            [ 75%]
tests/test_aut

### CUDA Kernel Benchmarks


In [ ]:
# Benchmark softmax kernels
!python kernel_tests/test_softmax_fw.py

# Benchmark layernorm kernels
!python kernel_tests/test_layernorm_fw.py


---

## Machine Translation Training

Train a transformer on the IWSLT14 German-English dataset.


In [ ]:
# Train without fused kernels (baseline)
# !python project/run_machine_translation.py --use-fused-kernel False

# Train with fused kernels (faster)
# !python project/run_machine_translation.py --use-fused-kernel True

print("Training command examples:")
print("  Without fused kernels: python project/run_machine_translation.py --use-fused-kernel False")
print("  With fused kernels:    python project/run_machine_translation.py --use-fused-kernel True")
print("\nNote: Training takes time. Uncomment above to actually run.")


---

## Summary

This notebook demonstrated the complete MiniTorch framework:

### Assignment 1: CUDA Tensor Operations
- Custom CUDA kernels for map, zip, reduce, and matrix multiplication
- Stride-based indexing for flexible tensor layouts
- Shared memory optimization for matrix multiplication

### Assignment 2: Automatic Differentiation
- Computation graph construction and backpropagation
- Efficient gradient computation via topological sort
- Neural network module abstractions

### Assignment 3: Transformer Architecture
- Multi-head attention mechanism
- Transformer layers with residual connections
- Full decoder-only language model
- Machine translation application

### Assignment 4: Optimized CUDA Kernels
- Fused softmax with attention mask support
- Fused layernorm with concurrent mean/variance computation
- Significant kernel-level speedups (up to 15.8×)
- Integrated into transformer for end-to-end acceleration

The framework demonstrates the complete pipeline from low-level CUDA programming to high-level transformer training, achieving production-quality performance through careful optimization.
